# Coflect HILT Workflow (Torch)

This notebook follows a standard deep learning experiment flow with Coflect integration:
1. Imports and reproducibility
2. Data loading and visualization
3. Model/loss/optimizer setup
4. Training
5. Evaluation


## 1) Imports and reproducibility


In [ ]:
from __future__ import annotations

import random
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.models import resnet18

from coflect.backends import TorchAdapter
from coflect.modules.hilt.common.torch_dataset import DatasetConfig, build_torch_dataset

from coflect.modules.hilt.common.notebook_bridge import (
    NotebookBridgeConfig,
    NotebookHILTBridge,
    roi_norm_to_pixels,
)


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 2) Data loading and visualization


In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    data_root: str = "./data"
    batch_size: int = 64
    num_workers: int = 2
    epochs: int = 3
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    download_data: bool = True
    snapshot_dir: str = "./snapshots"
    snapshot_every: int = 50
    xai_every: int = 25


cfg = ExperimentConfig()
print(cfg)

train_ds = build_torch_dataset(
    DatasetConfig(
        name="cifar10_catsdogs",
        root=cfg.data_root,
        split="train",
        download_data=cfg.download_data,
    )
)

val_ds = build_torch_dataset(
    DatasetConfig(
        name="cifar10_catsdogs",
        root=cfg.data_root,
        split="test",
        download_data=cfg.download_data,
    )
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

snapshot_dir = Path(cfg.snapshot_dir)
snapshot_dir.mkdir(parents=True, exist_ok=True)

print("train samples:", len(train_ds), "val samples:", len(val_ds))


In [ ]:
images, labels, sample_ids = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
axes = axes.flatten()
for i in range(8):
    img = images[i].permute(1, 2, 0).clamp(0, 1).cpu().numpy()
    label = int(labels[i].item())
    sample_id = int(sample_ids[i].item())
    class_name = "dog" if label == 1 else "cat"

    axes[i].imshow(img)
    axes[i].set_title(f"{class_name} (idx={sample_id})")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## 3) Model, loss, optimizer, and Coflect adapter setup


In [ ]:
model = resnet18(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
adapter = TorchAdapter(model=model, optimizer=optimizer, criterion=criterion)

bridge = NotebookHILTBridge(
    NotebookBridgeConfig(
        server="http://127.0.0.1:8000",
        backend="torch",
        metrics_every=10,
        feedback_every=10,
        xai_every=cfg.xai_every,
        auto_start_backend=True,
        auto_start_xai_worker=True,
        backend_host="127.0.0.1",
        backend_port=8000,
        xai_snapshot_dir=str(snapshot_dir),
        xai_dataset="cifar10_catsdogs",
        data_root=cfg.data_root,
        split="train",
        download_data=cfg.download_data,
        xai_method="consensus",
        log_dir="./.coflect_logs/notebook_bridge",
    )
)

print("model parameters:", sum(p.numel() for p in model.parameters()))
print("bridge auto-start enabled: backend + XAI worker should launch automatically")
print("open UI at http://127.0.0.1:8000")


## 4) Training

This loop also emits compact metrics and periodic XAI requests to the backend.
To see the Live panel update, keep backend + XAI worker running.


In [ ]:
@torch.no_grad()
def evaluate(model: torch.nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> tuple[float, float]:
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for x, y, _ in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)
        preds = logits.argmax(dim=1)

        total_correct += int((preds == y).sum().item())
        total_seen += int(y.size(0))
        total_loss += float(loss.detach().item()) * int(y.size(0))

    return total_loss / max(1, total_seen), total_correct / max(1, total_seen)


In [ ]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "epoch_time_sec": [],
}

global_step = 0
focus_lambda = bridge.focus_lambda
roi_norm = bridge.roi_norm

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    model.train()

    train_total_loss = 0.0
    train_total_correct = 0
    train_total_seen = 0

    for x, y, sample_ids in train_loader:
        global_step += 1

        update = bridge.maybe_sync_feedback(step=global_step)
        if update.changed:
            focus_lambda = update.focus_lambda
            roi_norm = update.roi_norm

        while bridge.paused:
            time.sleep(0.25)
            pause_update = bridge.maybe_sync_feedback(step=global_step, force=True)
            if pause_update.changed:
                focus_lambda = pause_update.focus_lambda
                roi_norm = pause_update.roi_norm

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = adapter.forward(x)
        primary_loss = adapter.loss(logits, y)
        total_loss = primary_loss

        if focus_lambda > 0.0 and roi_norm is not None:
            pixels = roi_norm_to_pixels(roi_norm, height=int(x.shape[2]), width=int(x.shape[3]))
            if pixels is not None:
                x0, y0, x1, y1 = pixels
                mask = torch.zeros((x.size(0), 1, x.size(2), x.size(3)), device=device)
                mask[:, :, y0:y1, x0:x1] = 1.0
                masked_x = x * mask
                aux_logits = adapter.forward(masked_x)
                aux_loss = adapter.loss(aux_logits, y)
                total_loss = total_loss + float(focus_lambda) * aux_loss

        adapter.step(total_loss)

        with torch.no_grad():
            preds = logits.argmax(dim=1)
            batch_acc = float((preds == y).float().mean().item())

        train_total_seen += int(y.size(0))
        train_total_correct += int((preds == y).sum().item())
        train_total_loss += float(primary_loss.detach().item()) * int(y.size(0))

        sample_idx0 = int(sample_ids[0].item())
        target0 = int(y[0].item())
        pred0 = int(preds[0].item())
        bridge.maybe_enqueue_xai(
            step=global_step,
            sample_idx=sample_idx0,
            target_class=target0,
            pred_class=pred0,
            request_kind="periodic",
            force=(global_step == 1),
        )

        if global_step == 1 or global_step % max(1, cfg.snapshot_every) == 0:
            tmp_path = snapshot_dir / f".model_step_{global_step:06d}.pt.tmp"
            final_path = snapshot_dir / f"model_step_{global_step:06d}.pt"
            torch.save(model.state_dict(), tmp_path)
            tmp_path.replace(final_path)

        bridge.maybe_post_metrics(
            step=global_step,
            loss=float(primary_loss.detach().item()),
            acc=batch_acc,
            epoch=float(epoch - 1) + (train_total_seen / max(1, len(train_ds))),
            focus_lambda=float(focus_lambda),
        )

    train_loss = train_total_loss / max(1, train_total_seen)
    train_acc = train_total_correct / max(1, train_total_seen)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    dt = time.time() - t0

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["epoch_time_sec"].append(dt)

    bridge.maybe_post_metrics(
        step=global_step,
        loss=float(train_loss),
        acc=float(train_acc),
        epoch=float(epoch),
        focus_lambda=float(focus_lambda),
        extra={"val_loss": float(val_loss), "val_acc": float(val_acc)},
        force=True,
    )

    print(
        f"epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"focus={focus_lambda:.3f} | "
        f"time={dt:.1f}s"
    )


## 5) Evaluation and plots


In [ ]:
epochs = np.arange(1, cfg.epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
axes[0].plot(epochs, history["val_loss"], marker="o", label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs, history["train_acc"], marker="o", label="train")
axes[1].plot(epochs, history["val_acc"], marker="o", label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
final_val_loss, final_val_acc = evaluate(model, val_loader, criterion, device)
print(f"final validation loss: {final_val_loss:.4f}")
print(f"final validation accuracy: {final_val_acc:.4f}")
